specific to uh tide gauges, different format and files

In [1]:
from bs4 import BeautifulSoup
import csv
import sys
import os

from datetime import datetime, timedelta
from string import Template

In [2]:
def round_to_min(dt):
    """Rounds a datetime object to the nearest min."""
    if dt.second >= 30:
        return dt.replace(second=0, microsecond=0) + timedelta(minutes=1)
    else:
        return dt.replace(second=0, microsecond=0)

In [3]:
def html_table_to_string(html_table):
    """
    Converts an HTML table to a string representation.

    Args:
        html_table: A string containing the HTML table.

    Returns:
        A string representation of the table.
    """
    soup = BeautifulSoup(html_table, 'html.parser')
    table = soup.find('table')

    if table is None:
        return "No table found in the HTML."

    table_string = ""
    for row in table.find_all('tr'):
        row_string = ""
        for cell in row.find_all(['th', 'td']):
            row_string += cell.get_text(strip=True) + "\t"  # Use tab as a separator
        table_string += row_string.rstrip("\t") + "\n"  # Remove trailing tab and add newline
    return table_string.rstrip("\n")

In [4]:
def parse_table(table):
    soup = BeautifulSoup(table, 'html.parser')

    date_format = '%Y-%m-%d %H:%M:%S'
    #table_dct={}
    tg_dt=[]
    tg_lvl=[]

    content_idx=2 # default to third entry for lvl info
    for row in soup.find_all('tr'):
        row_content=row.find_all(['th', 'td'])
        # skip first row
        if row_content[0].string.startswith('Time'):
            # check the name of the third entry - if pressure, use fourth entry (radar)
            #print(row_content[0])
            #print(row_content[0].string.startswith('Time'))
            #print(row_content[2])
            #print(row_content[2].string.startswith('prs'))
            if row_content[2].string.startswith('prs'):
                content_idx=3
            #print(content_idx)
            continue
        # time and relative height in m
        #table_dct[row_content[0].string]=row_content[content_idx].string
        #print(row_content)
        #print(row_content[0].string+' '+row_content[1].string+' '+row_content[2].string+' '+row_content[3].string)
        #print(row_content[content_idx].string)
        tg_dt_str=row_content[0].string
        tg_dt.append(datetime.strptime(tg_dt_str,date_format))
        tg_lvl.append(row_content[content_idx].string)
    #print(table_dct)
    return tg_dt, tg_lvl
    #return(table_dct)

In [5]:
filepath_tg='/Users/sahuang/Documents/DATA/tg/'
areas=['ofuas','tauas','auasi','aunuu']
#areas=['tauas','auasi','aunuu']
#areas=['auasi','aunuu']

date_format = '%m/%d/%y %H:%M'#:%S'

#aoi_names=["Aunu'u",'Ofu','Olosega',"Ta'u"] # relevant aoi names for the TGs
suffix='.txt'
filetype='*'+suffix

tg_files=[]

results_header='        AREA          |   DATE AND TIME      |   TG DATE AND TIME   | TG LEVEL (M)'
results_template=Template('${AREA}     ${DT}    ${TG_DT}       ${TG_LVL}')

filepath='/Users/sahuang/Documents/Notes/CSDA - Umbra 24-25/Tide Gauge/'
times_filename='Capture_Times.csv'
csv_file=filepath+times_filename

for area in areas:
    # Read data in for the tg station
    tg_dt_all=[]
    tg_lvl_all=[]
    dirfiles=os.listdir(filepath_tg+area+'/')
    print(area)

    for dirfile in dirfiles:
        if suffix in dirfile:# and dirfile.endswith('-06.txt'):
            tg_file=filepath_tg+area+'/'+dirfile
            tg_files.append(tg_file)
            #print(tg_file)
            
            with open(tg_file, 'r') as f:
                content=f.read()
                tg_dt,tg_lvl=parse_table(content)
                #s=html_table_to_string(content)
                #print(s)
                tg_dt_all=tg_dt_all+tg_dt
                tg_lvl_all=tg_lvl_all+tg_lvl
    #sys.exit()
    #print((len(tg_dt_all))
    #sys.exit()
            
    # Read in image times
    match area:
        case 'ofuas':
            aoi_names=['Ofu','Olosega']
        case 'tauas':
            aoi_names=["Ta'u"]
        case 'auasi':
            aoi_names=["Aunu'u"]
        case 'aunuuu':
            aoi_names=["Aunu'u"]

    #print(aoi_names)
    print(results_header)
    #sys.exit()
    for aoi_name in aoi_names:
        with open(csv_file, 'r',encoding='utf-8-sig') as csvf:
            csvr = csv.DictReader(csvf)
            for row in csvr:
                if not row[aoi_name]:
                    continue
                else:
                    aoi_dt_i=datetime.strptime(row[aoi_name],date_format)
                    #idx=tg_dt_all.index(round_to_min(aoi_dt_i))
                    closest_datetime = min(tg_dt_all, key=lambda x: abs(x - round_to_min(aoi_dt_i)))
                    #print('Closest Datetime ',closest_datetime)
                    idx=tg_dt_all.index(closest_datetime)
                    #print(
                
                    # format for printing
                    results_dct={}
                    results_dct['AREA']=f"{aoi_name:<20}"
                    results_dct['DT']=aoi_dt_i
                    results_dct['TG_DT']=tg_dt_all[idx]
                    results_dct['TG_LVL']=f"{tg_lvl_all[idx]:<5}"
                    print(results_template.substitute(**results_dct))
        #sys.exit()

ofuas
        AREA          |   DATE AND TIME      |   TG DATE AND TIME   | TG LEVEL (M)
Ofu                      2024-10-18 21:37:00    2024-10-18 21:37:00       1.589
Ofu                      2024-10-25 10:05:00    2024-10-25 10:05:00       1.606
Ofu                      2024-11-14 21:17:00    2024-11-14 21:17:00       1.137
Ofu                      2024-11-21 20:15:00    2024-11-21 20:15:00       1.457
Ofu                      2024-12-19 20:26:00    2024-12-19 20:26:00       1.801
Ofu                      2024-12-24 08:24:00    2024-12-24 08:24:00       1.42 
Ofu                      2025-01-06 21:32:00    2025-01-06 21:30:00       1.798
Ofu                      2025-01-07 21:33:00    2025-01-07 21:30:00       1.63 
Ofu                      2025-01-12 20:13:00    2025-01-12 20:13:00       1.515
Ofu                      2025-02-13 10:07:00    2025-02-13 10:07:00       1.53 
Ofu                      2025-02-15 21:25:00    2025-02-15 21:25:00       1.869
Ofu                      2025-0